In [133]:
df=pd.read_csv('data/cville_weather_full.csv')

/var/folders/b1/znq25z59401dlz_b0y0j_2c00000gn/T/ipykernel_38532/169479424.py:1: DtypeWarning: Columns (17,19,21,23,27,29,31,35,37,39,41,43,45,47) have mixed types. Specify dtype option on import or set low_memory=False.
  df=pd.read_csv('data/cville_weather_full.csv')


In [134]:
df.columns
df.head()

,STATION,DATE,LATITUDE,LONGITUDE,ELEVATION,NAME,PRCP,PRCP_ATTRIBUTES,SNOW,SNOW_ATTRIBUTES,...,WT08,WT08_ATTRIBUTES,WT11,WT11_ATTRIBUTES,WT14,WT14_ATTRIBUTES,WT16,WT16_ATTRIBUTES,WT18,WT18_ATTRIBUTES
0,USC00441593,1893-01-01,38.0329,-78.5226,264,"CHARLOTTESVILLE 2 W, VA US",399.0,",,6,",NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,",,6"
1,USC00441593,1893-01-02,38.0329,-78.5226,264,"CHARLOTTESVILLE 2 W, VA US",NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,USC00441593,1893-01-03,38.0329,-78.5226,264,"CHARLOTTESVILLE 2 W, VA US",NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,USC00441593,1893-01-04,38.0329,-78.5226,264,"CHARLOTTESVILLE 2 W, VA US",66.0,",,6,",NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,USC00441593,1893-01-05,38.0329,-78.5226,264,"CHARLOTTESVILLE 2 W, VA US",NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [135]:
cols_to_keep = ["STATION", "DATE", "NAME", "PRCP", "SNOW", "SNWD", "TMAX", "TMIN", "WT01", "WT03", "WT04", 
                "WT05", "WT06", "WT08", "WT11", "WT14", "WT16", "WT18"]

df = df[cols_to_keep]

In [136]:
df["DATE"] = pd.to_datetime(df["DATE"], format="mixed", errors="coerce")
df = df.dropna(subset=["DATE"])

df = df[(df["DATE"] >= "2000-01-01") & (df["DATE"] <= "2025-12-31")]

df = df.sort_values("DATE")

In [137]:
print(df["DATE"].min())
print(df["DATE"].max())
print(len(df))

2000-01-01 00:00:00
2025-12-31 00:00:00
9437


In [138]:
df["TMAX"] = df["TMAX"] / 10.0
df["TMIN"] = df["TMIN"] / 10.0

In [139]:
df["TMAX"] = (df["TMAX"] * 1.8) + 32
df["TMIN"] = (df["TMIN"] * 1.8) + 32

In [141]:
df["TMAX"] = df["TMAX"].round(1)
df["TMIN"] = df["TMIN"].round(1)

In [142]:
for col in ["TMAX", "TMIN", "PRCP"]:
    df[col] = df[col].fillna(df[col].median())

In [144]:
for col in ["SNOW", "SNWD"]:
    df[col] = df[col].fillna(0)

In [145]:
wt_cols = ["WT01","WT03","WT04","WT05","WT06","WT08","WT11","WT14","WT16","WT18"]
df[wt_cols] = df[wt_cols].replace(r'^\s*$', None, regex=True)
df[wt_cols] = df[wt_cols].apply(pd.to_numeric, errors='coerce')
df[wt_cols] = df[wt_cols].fillna(0).astype(int)

In [107]:
df["PRCP"].describe()

count    9437.000000
mean       32.897001
std        93.433433
min         0.000000
25%         0.000000
50%         0.000000
75%        13.000000
max      1765.000000
Name: PRCP, dtype: float64

In [108]:
df["TMAX"].describe()

count    9437.000000
mean       67.744643
std        17.616205
min        17.100000
25%        54.000000
50%        70.000000
75%        82.900000
max       105.100000
Name: TMAX, dtype: float64

In [109]:
def classify_weather(row):
    if row["WT06"] == 1:
        return "Ice"
    
    if row["WT04"] == 1:
        return "Sleet"
    
    if row["WT18"] == 1:
        return "Snow"
    
    if row["WT03"] == 1:
        return "Thunder"
    
    if row["WT05"] == 1:
        return "Hail"
    
    if row["WT16"] == 1 or row["PRCP"] > 0:
        if row["PRCP"] > 13:
            return "Heavy-Rain"
        elif row["WT14"] == 1:
            return "Drizzle"
        elif row["TMAX"] > 83:
            return "Hot-Rainy"
        elif row["TMIN"] < 54:
            return "Cold-Rainy"
        else:
            return "Mild-Rainy"
    
    # --- Priority 4: Wind (only if nothing else) ---
    if row["WT11"] == 1:
        return "Windy"
    
    # --- Priority 5: Fog ---
    if row["WT01"] == 1:
        return "Fog"
    
    # --- Priority 6: Dry ---
    if row["TMAX"] > 83:
        return "Hot-Dry"
    elif row["TMIN"] < 54:
        return "Cold-Dry"
    else:
        return "Mild-Dry"

In [110]:
df["STATE"] = df.apply(classify_weather, axis=1)

In [111]:
df.to_csv("cville_weather_cleaned.csv", index=False)